# Aula 4 — Inferência na Regressão

**Laboratórios de Econometria em R** · MPEF/FGV-EPGE · Prof. Marcelo Mello

---

Estimar não basta: é preciso medir a **incerteza**. Quatro resultados:

1. a tabela completa de inferência — estatística $t$, valor-$p$, intervalo de confiança;
2. o que "95% de confiança" **significa**, verificado por simulação;
3. **heterocedasticidade**: o que ela estraga (a inferência) e o que não estraga (o estimador);
4. regressão com *dummy* **é** diferença de médias.

> Os pacotes `sandwich` e `AER` não são necessários aqui: implementamos o erro-padrão
> robusto (HC1) a partir da fórmula, o que deixa explícito o que o software faz por
> baixo do capô.

In [ ]:
for (p in c("ggplot2", "dplyr")) {
  if (!requireNamespace(p, quietly = TRUE)) install.packages(p, quiet = TRUE)
}
library(ggplot2)
suppressMessages(library(dplyr))
theme_set(theme_minimal(base_size = 13))
az <- "#1f4e79"; vd <- "#2e7d32"; vm <- "#b3261e"; cz <- "grey45"
options(repr.plot.width = 8, repr.plot.height = 4)

## 0. As duas funções que usaremos o curso inteiro

O erro-padrão robusto a heterocedasticidade (HC1) é o **padrão da prática aplicada**.
A fórmula "sanduíche" é
$(X'X)^{-1}\left[X'\,\text{diag}(\hat u^2)\,X\right](X'X)^{-1}$,
com a correção $n/(n-k)$ da variante HC1.

In [ ]:
ep_robusto <- function(modelo, tipo = "HC1") {
  X <- model.matrix(modelo); u <- residuals(modelo)
  n <- nrow(X); k <- ncol(X)
  bread  <- solve(crossprod(X))          # (X'X)^{-1}
  ajuste <- if (tipo == "HC1") n / (n - k) else 1
  meat   <- crossprod(X * u)             # X' diag(u^2) X
  sqrt(diag(bread %*% (ajuste * meat) %*% bread))
}

tabela_reg <- function(modelo, robusto = TRUE) {
  b  <- coef(modelo)
  ep <- if (robusto) ep_robusto(modelo) else summary(modelo)$coefficients[, 2]
  t  <- b / ep
  gl <- df.residual(modelo)
  data.frame(coef = b, ep = ep, t = t,
             p = 2 * pt(-abs(t), gl),
             ic_inf = b - qt(0.975, gl) * ep,
             ic_sup = b + qt(0.975, gl) * ep)
}

In [ ]:
set.seed(5490)
n_d <- 420
beta0_v <- 698.9; beta1_v <- -2.28

STR       <- rnorm(n_d, mean = 19.64, sd = 1.89)
TestScore <- beta0_v + beta1_v * STR + rnorm(n_d, 0, 18.6)
ca <- data.frame(TestScore, STR)

m1 <- lm(TestScore ~ STR, data = ca)
round(tabela_reg(m1), 4)

Lendo a linha do `STR`: a estatística $t$ testa $H_0:\beta_1 = 0$; o valor-$p$ é o
menor nível ao qual se rejeita; e o intervalo de 95% não contém o zero — coerente com
a rejeição.

## 1. O que significa "95% de confiança"

A interpretação correta é sobre o **procedimento**, não sobre um intervalo particular:
se repetíssemos a amostragem muitas vezes, cerca de 95% dos intervalos construídos
conteriam o parâmetro verdadeiro. Vamos contar.

In [ ]:
set.seed(99)
M <- 2000
cobre <- replicate(M, {
  x  <- rnorm(n_d, 19.64, 1.89)
  yy <- beta0_v + beta1_v * x + rnorm(n_d, 0, 18.6)
  fit <- lm(yy ~ x)
  ep  <- summary(fit)$coefficients[2, 2]
  ic  <- coef(fit)[2] + c(-1, 1) * qt(0.975, df.residual(fit)) * ep
  ic[1] <= beta1_v && beta1_v <= ic[2]
})

c(nominal = 95, cobertura_simulada = 100 * mean(cobre))

Perto de 95%, como prometido — **sob homocedasticidade**, que é o caso desta
simulação. A próxima seção mostra o que acontece quando essa hipótese falha.

## 2. Heterocedasticidade

Quando a variância do erro depende de $X$, os erros-padrão **convencionais** ficam
errados. Note que o estimador $\hat\beta_1$ continua não viesado — o que quebra é a
**inferência**.

In [ ]:
set.seed(404)
M <- 2000; n <- 200
res <- t(replicate(M, {
  x  <- runif(n, 10, 30)
  # desvio do erro CRESCE com x: heterocedasticidade forte
  yy <- 700 - 2.3 * x + rnorm(n, 0, 2 + 1.6 * (x - 10))
  fit <- lm(yy ~ x)
  ep_conv <- summary(fit)$coefficients[2, 2]
  ep_rob  <- unname(ep_robusto(fit)[2])
  gl <- df.residual(fit)
  b1 <- unname(coef(fit)[2])
  c(b1       = b1,
    cob_conv = abs(b1 - (-2.3)) <= qt(0.975, gl) * ep_conv,
    cob_rob  = abs(b1 - (-2.3)) <= qt(0.975, gl) * ep_rob)
}))

round(c(beta1_verdadeiro   = -2.3,
        media_estimada     = mean(res[, "b1"]),
        vies               = mean(res[, "b1"]) - (-2.3),
        cobertura_convencional = 100 * mean(res[, "cob_conv"]),
        cobertura_robusta      = 100 * mean(res[, "cob_rob"])), 3)

O resultado é a lição da aula:

- o **viés é praticamente nulo** — heterocedasticidade não vicia o estimador;
- a cobertura do intervalo **convencional** fica longe dos 95% prometidos;
- a cobertura do intervalo **robusto** fica próxima do alvo.

> A garantia do erro-padrão robusto é **assintótica**: em amostras pequenas a
> cobertura ainda pode ficar um pouco abaixo do nominal. Ele conserta o problema
> principal, não faz milagre.

## 3. Regressão com *dummy* é diferença de médias

Este resultado reconcilia o instrumento (regressão) com o arcabouço experimental da
Aula 1.

In [ ]:
set.seed(7)
grupo <- rbinom(n_d, 1, 0.45)                       # 0 = controle, 1 = tratado
y_d   <- 640 + 12 * grupo + rnorm(n_d, 0, 18)
dd    <- data.frame(y_d, grupo)

m_dummy <- lm(y_d ~ grupo, data = dd)

c(intercepto      = coef(m_dummy)[1],
  media_grupo0    = mean(y_d[grupo == 0]),
  coef_dummy      = coef(m_dummy)[2],
  diferenca_medias = mean(y_d[grupo == 1]) - mean(y_d[grupo == 0])) |> round(6)

O intercepto **é** a média do grupo de controle, e o coeficiente da *dummy* **é** a
diferença entre as médias. Não é analogia: é identidade algébrica.

In [ ]:
# e o teste t da regressão equivale ao teste de diferença de médias
t_reg   <- summary(m_dummy)$coefficients[2, 3]
t_welch <- t.test(y_d[grupo == 1], y_d[grupo == 0], var.equal = TRUE)$statistic

c(t_da_regressao = t_reg, t_de_duas_amostras = as.numeric(t_welch)) |> round(6)

## Para experimentar

1. Na seção 2, zere a heterocedasticidade (`rnorm(n, 0, 12)`) e verifique que as duas
   coberturas voltam a ~95%: o erro-padrão robusto **não custa nada** quando não é
   preciso.
2. Aumente `n` para 2000 na mesma seção e veja a cobertura robusta se aproximar de 95%.
3. Use `var.equal = FALSE` no `t.test` e compare com o $t$ robusto da regressão.

---

⬅️ [Aula 3](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/03-regressao-simples.ipynb) · [🏠 Índice](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/00-indice.ipynb) · ➡️ [**Aula 5 — Regressão múltipla**](https://colab.research.google.com/github/vitorwilher/doutorado-epge/blob/main/labs/econometria/05-regressao-multipla.ipynb)